# Step 5: Feature Engineering
Calculates business features: `total_order_value`, `delivery_days`, `delivery_delay`, `customer_order_count`, `average_order_value`, `seller_revenue`, and `repeat_customer_indicator`.


In [ ]:
import pandas as pd
import sqlite3
import os

import sys
import os

def resolve_path(rel_path):
    curr = os.path.abspath(os.getcwd())
    while curr and os.path.dirname(curr) != curr:
        candidate = os.path.join(curr, rel_path)
        if os.path.exists(candidate):
            return os.path.abspath(candidate)
        curr = os.path.dirname(curr)
    return os.path.abspath(rel_path)

db_path = resolve_path("data/cleaned/ecommerce.db")
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA temp_store = MEMORY;")
orders = pd.read_sql("SELECT * FROM orders", conn)
order_items = pd.read_sql("SELECT * FROM order_items", conn)

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_delay'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['is_delayed'] = orders['delivery_delay'] > 0

print("Engineered features preview:", flush=True)
print(orders[['order_id', 'delivery_days', 'delivery_delay', 'is_delayed']].head(), flush=True)
conn.close()

